In [ ]:
!rm -r *
# This command builds the special URL and passes it to aria2c
# -c = continue download
# -x 16 = max 16 connections per download
# -s 16 = split file into 16 pieces
# -k 1M = min split size
# -o = output file name
!apt install aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://zenodo.org/records/17252365/files/cadenza_clip1_data.train.v1.0.tar.gz?download=1" -o cadenza_clip1_data.train.v1.0.tar.gz

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libaria2-0 libc-ares2
The following NEW packages will be installed:
  aria2 libaria2-0 libc-ares2
0 upgraded, 3 newly installed, 0 to remove and 165 not upgraded.
Need to get 1,468 kB/1,513 kB of archives.
After this operation, 5,441 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libaria2-0 amd64 1.36.0-1 [1,086 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 aria2 amd64 1.36.0-1 [381 kB]
Fetched 786 kB in 2s (452 kB/s)

78Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 128639 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
7Progress: [  0%] [..........................................................] 87Progress: [  8%] [####......................................

In [ ]:
# This will extract the files into the current directory
!tar -xzvf cadenza_clip1_data.train.v1.0.tar.gz
!rm cadenza_clip1_data.train.v1.0.tar.gz
!ls -l cadenza_data 
!ls -l cadenza_data/metadata
!ls -l cadenza_data/train


In [ ]:
# Create the directory (ignore error if it already exists)
!mkdir -p baseline

# Run npx with -y to auto-confirm, and tell degit to download INTO the baseline folder
!npx -y degit --force claritychallenge/clarity/recipes/cad_icassp_2026/baseline#main baseline

In [ ]:
import os
import sys

# All subsequent commands will be run from here.
os.chdir('baseline')
print(f"Current working directory: {os.getcwd()}")

# --- Install packages ---
print("\nInstalling required packages...")

# Install standard packages
!pip install -q hydra-core omegaconf pandas pystoi demucs openai-whisper jiwer inflect

# Install the pyclarity library directly from the official GitHub repository
print("\nInstalling pyclarity library from GitHub...")
!pip install -q git+https://github.com/claritychallenge/clarity.git

print("\nEnvironment setup complete.")

In [ ]:
import os

# just making this more structured
exp_dir = "exp"
os.makedirs(exp_dir, exist_ok=True)

local_precomputed_path = "precomputed"

# --- Copy STOI scores ---
print("Copying STOI scores...")
!cp {local_precomputed_path}/cadenza_data.train.stoi.jsonl {exp_dir}/
!cp {local_precomputed_path}/cadenza_data.valid.stoi.jsonl {exp_dir}/

# --- Copy Whisper scores ---
print("Copying Whisper scores...")
!cp {local_precomputed_path}/cadenza_data.train.whisper.jsonl {exp_dir}/
!cp {local_precomputed_path}/cadenza_data.valid.whisper.jsonl {exp_dir}/

print("\nPre-computed scores copied to the 'exp/' directory.")
!ls -l {exp_dir}

In [ ]:
import os
# just downloading the validation data.
os.chdir('/kaggle/working')

print("--- Downloading Validation Data ---")
# This is the same Zenodo record as the training data, just a different file
!aria2c -c -x 16 -s 16 -k 1M -o cadenza_clip1_data.valid.v1.0.tar.gz "https://zenodo.org/records/17252365/files/cadenza_clip1_data.valid.v1.0.tar.gz"

print("\n--- Extracting Validation Data ---")
!tar -xzvf cadenza_clip1_data.valid.v1.0.tar.gz
!rm cadenza_clip1_data.valid.v1.0.tar.gz

print("\nValidation data is now in place.")
!ls -l cadenza_data/metadata

# Return to our main working directory for the baseline code
os.chdir('baseline')
print(f"\nReturned to directory: {os.getcwd()}")

In [ ]:
# some key issue fixing
print("--- Inspecting first line of the whisper score file ---")
!head -n 1 exp/cadenza_data.train.whisper.jsonl

# The key is "whisper_mixture". We need to replace it with "whisper".
print('\n--- Correcting the key in whisper.jsonl files ---')
!sed -i 's/"whisper.mixture"/"whisper"/g' exp/cadenza_data.train.whisper.jsonl
!sed -i 's/"whisper.mixture"/"whisper"/g' exp/cadenza_data.valid.whisper.jsonl

print("Keys have been corrected. Verifying the change:")
!head -n 1 exp/cadenza_data.train.whisper.jsonl

In [ ]:
# --- Run STOI baseline prediction ---
print("--- Generating predictions for STOI baseline ---")
!python predict.py baseline=stoi data.cadenza_data_root=/kaggle/working
print("\nSTOI prediction complete.")

# --- Run Whisper baseline prediction ---
print("\n--- Generating predictions for Whisper baseline ---")
!python predict.py baseline=whisper data.cadenza_data_root=/kaggle/working
print("\nWhisper prediction complete.")

In [ ]:
print('\n--- Previewing STOI Predictions ---')
!head exp/cadenza_data.stoi.valid.predict.csv

print('\n--- Previewing Whisper Predictions ---')
!head exp/cadenza_data.whisper.valid.predict.csv

#### Now we shall be trying to run whisper large v3 turbo as a baseline too for further analysis

In [ ]:
%%bash
# Create the audio directory if it doesn't exist
mkdir -p /kaggle/working/cadenza_data/audio

# Remove any existing broken symlinks (ignore errors if they don't exist)
rm -f /kaggle/working/cadenza_data/audio/train
rm -f /kaggle/working/cadenza_data/audio/valid

# Create correct symlinks pointing to the actual data location
ln -s /kaggle/working/cadenza_data/train /content/cadenza_data/audio/train
ln -s /kaggle/working/cadenza_data/valid /content/cadenza_data/audio/valid

# Verify the new symlinks
echo "Verifying new symlinks:"
ls -la /kaggle/working/cadenza_data/audio/
echo ""
echo "Checking if we can access the files:"
ls /kaggle/working/cadenza_data/audio/train/signals/ | head -5

In [ ]:
# This ensures both commands run in the same shell.
# First, change directory to where the script is located.
# Then, execute the python script with its arguments.
!cd /kaggle/working/baseline/ && python compute_whisper.py \
  split=train \
  data.cadenza_data_root=/kaggle/working/ \
  baseline=whisper \
  baseline.whisper_version=large-v3-turbo \
  baseline.system=whisper-large-v3-turbo

In [ ]:
!cd /kaggle/working/baseline/ && python compute_whisper.py \
  split=valid \
  data.cadenza_data_root=/kaggle/working/ \
  baseline=whisper \
  baseline.whisper_version=large-v3-turbo \
  baseline.system=whisper-large-v3-turbo

In [ ]:
# You should see 'cadenza_data.train.whisper_large.jsonl' and 
# 'cadenza_data.valid.whisper_large.jsonl' in the output list.
!ls -lh /kaggle/working/baseline/exp/

In [ ]:
# Execute predict.py using the scores from your new 'whisper-large-v3-turbo' system.
# Note that the split is 'valid' because we are making predictions on the validation set.
!cd /kaggle/working/baseline/ && python predict.py \
  split=valid \
  data.cadenza_data_root=/kaggle/working/ \
  baseline=whisper \
  baseline.system=whisper-large-v3-turbo

In [ ]:
# now you should see the new prediction file.
!ls -lh /kaggle/working/baseline/exp/

In [ ]:
# Run the evaluation script on the prediction file so we now know the rmse and corelation
!cd /kaggle/working/baseline/ && python evaluate.py \
  split=valid \
  data.cadenza_data_root=/content/ \
  baseline=whisper \
  baseline.system=whisper-large-v3-turbo

## Training

In [13]:
# Cell 1: Install Dependencies
# Install aria2 for accelerated downloading
!apt-get update && apt-get install -y aria2

# CRITICAL FIX: Install specific library versions to avoid conflicts
# The protobuf==3.20.3 is the key to fixing the model loading error.
!pip install -q transformers datasets accelerate torch pandas librosa soundfile "protobuf==3.20.3" sentencepiece

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,452 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,827 kB]
Fetched 12.5 MB in 5s (2,567 kB/s)          

In [5]:
# Cell 2: Download and Extract Data
import os

# --- Configuration ---
WORKING_DIR = "/kaggle/working"
TRAIN_URL = "https://zenodo.org/records/17252365/files/cadenza_clip1_data.train.v1.0.tar.gz?download=1"
VALID_URL = "https://zenodo.org/records/17252365/files/cadenza_clip1_data.valid.v1.0.tar.gz?download=1"
TRAIN_ARCHIVE_NAME = "cadenza_clip1_data.train.v1.0.tar.gz"
VALID_ARCHIVE_NAME = "cadenza_clip1_data.valid.v1.0.tar.gz"
DATA_DIR = os.path.join(WORKING_DIR, "cadenza_data")

# --- Download ---
print("Downloading training data... (This can take several minutes)")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{TRAIN_URL}" -d "{WORKING_DIR}" -o "{TRAIN_ARCHIVE_NAME}"
print("Training data downloaded.")

print("\nDownloading validation data...")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{VALID_URL}" -d "{WORKING_DIR}" -o "{VALID_ARCHIVE_NAME}"
print("Validation data downloaded.")

# --- Extract ---
os.makedirs(DATA_DIR, exist_ok=True)
print("\nExtracting training data...")
# Use --strip-components=1 to remove the top-level folder from the archive
!tar -xzf {os.path.join(WORKING_DIR, TRAIN_ARCHIVE_NAME)} --strip-components=1 -C {DATA_DIR}
print("Training data extracted.")

print("\nExtracting validation data...")
# Use --strip-components=1 here as well
!tar -xzf {os.path.join(WORKING_DIR, VALID_ARCHIVE_NAME)} --strip-components=1 -C {DATA_DIR}
print("Validation data extracted.")

# --- Cleanup ---
print("\nCleaning up downloaded archive files...")
!rm {os.path.join(WORKING_DIR, TRAIN_ARCHIVE_NAME)}
!rm {os.path.join(WORKING_DIR, VALID_ARCHIVE_NAME)}
print("Cleanup complete.")

print("\nFinal directory structure should now be correct:")
!ls -l {DATA_DIR}

 *** Download Progress Summary as of Sat Nov 15 10:49:44 2025 ***              3m43s]m0m
[#04dbe3 664MiB/4.2GiB(15%) CN:16 DL:12MiB ETA:5m]
FILE: /kaggle/working/cadenza_clip1_data.train.v1.0.tar.gz
-------------------------------------------------------------------------------

 *** Download Progress Summary as of Sat Nov 15 10:50:45 2025 ***              3m33s]
[#04dbe3 1.6GiB/4.2GiB(38%) CN:16 DL:14MiB ETA:2m57s]
FILE: /kaggle/working/cadenza_clip1_data.train.v1.0.tar.gz
-------------------------------------------------------------------------------

 *** Download Progress Summary as of Sat Nov 15 10:51:46 2025 ***              1m27s]
[#04dbe3 2.6GiB/4.2GiB(62%) CN:16 DL:18MiB ETA:1m27s]
FILE: /kaggle/working/cadenza_clip1_data.train.v1.0.tar.gz
-------------------------------------------------------------------------------

 *** Download Progress Summary as of Sat Nov 15 10:52:47 2025 ***              33s]mm
[#04dbe3 3.6GiB/4.2GiB(87%) CN:16 DL:16MiB ETA:32s]
FILE: /kaggle/working/

In [14]:
# Cell 3: Load and Prepare Data
from datasets import load_dataset, DatasetDict
import os

metadata_path = "/kaggle/working/cadenza_data/metadata/train_metadata.json"
audio_base_path = "/kaggle/working/cadenza_data/train/signals"

full_dataset = load_dataset('json', data_files=metadata_path, split='train')

def add_audio_path(example):
    example['audio_path'] = os.path.join(audio_base_path, f"{example['signal']}.flac")
    return example

full_dataset = full_dataset.map(add_audio_path)
split_dataset = full_dataset.train_test_split(test_size=0.1, seed=42)
split_dataset["validation"] = split_dataset.pop("test")

print("Dataset prepared and split:")
print(split_dataset)

Dataset prepared and split:
DatasetDict({
    train: Dataset({
        features: ['signal', 'prompt', 'response', 'n_words', 'words_correct', 'correctness', 'hearing_loss', 'audio_path'],
        num_rows: 7921
    })
    validation: Dataset({
        features: ['signal', 'prompt', 'response', 'n_words', 'words_correct', 'correctness', 'hearing_loss', 'audio_path'],
        num_rows: 881
    })
})


In [15]:
# Cell 4: Define the Custom PyTorch Dataset (Improved)
import torch
from torch.utils.data import Dataset
import librosa

class CadenzaDataset(Dataset):
    def __init__(self, hf_dataset, processor):
        self.hf_dataset = hf_dataset
        self.processor = processor
        self.hearing_loss_map = {"No Loss": 0, "Mild": 1, "Moderate": 2}

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        item = self.hf_dataset[idx]
        
        try:
            waveform, sr = librosa.load(item['audio_path'], sr=16000, mono=True)
        except Exception as e:
            print(f"Error loading {item['audio_path']}: {e}")
            return self.__getitem__((idx + 1) % len(self))
            
        features = self.processor(waveform, sampling_rate=16000, return_tensors="pt").input_features
        
        hearing_loss_label = torch.tensor(self.hearing_loss_map.get(item['hearing_loss'], 0), dtype=torch.long)
        correctness_score = torch.tensor([item['correctness']], dtype=torch.float32)

        # IMPROVEMENT: Also return the signal ID for submission generation
        return {
            "signal": item['signal'],
            "input_features": features.squeeze(0),
            "hearing_loss_label": hearing_loss_label,
            "correctness_score": correctness_score
        }

In [16]:
# Cell 5: Define the Whisper-SFM Model (Improved Regularization)
import torch.nn as nn
from transformers import WhisperProcessor, WhisperModel

class WhisperRegressionModel(nn.Module):
    def __init__(self, model_name="openai/whisper-large-v3-turbo", processor_name="openai/whisper-large-v3"):
        super().__init__()
        
        self.processor = WhisperProcessor.from_pretrained(processor_name)
        self.whisper = WhisperModel.from_pretrained(model_name, attn_implementation="sdpa")
        self.whisper.requires_grad_(False)
        self.whisper.eval()

        hidden_size = self.whisper.config.d_model
        embedding_dim = 32

        self.hearing_loss_embedding = nn.Embedding(num_embeddings=3, embedding_dim=embedding_dim)

        # IMPROVEMENT: Increased dropout rate for stronger regularization
        self.regression_head = nn.Sequential(
            nn.Linear(hidden_size + embedding_dim, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Dropout(0.5),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, input_features, hearing_loss_label):
        # IMPROVEMENT: Fixed deprecation warning
        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            hidden_states = self.whisper.encoder(input_features).last_hidden_state
        
        pooled_output = torch.mean(hidden_states, dim=1)
        hl_embedding = self.hearing_loss_embedding(hearing_loss_label)
        combined_features = torch.cat((pooled_output, hl_embedding), dim=1)
        
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            output = self.regression_head(combined_features)
            
        return output

In [17]:
# Cell 6: Set Up High-Efficiency & Robust Training (Improved)
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import gc

gc.collect()
torch.cuda.empty_cache()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# --- Configuration ---
MODEL_NAME = "openai/whisper-large-v3-turbo"
BATCH_SIZE = 8
ACCUMULATION_STEPS = 8
# IMPROVEMENT: Set num_workers to 0 to guarantee stability and prevent crashes.
NUM_WORKERS = 0 
LEARNING_RATE = 1e-4 # Starting LR, will be managed by scheduler
EPOCHS = 10 # Allow more epochs for early stopping and scheduler to work

model = WhisperRegressionModel(model_name=MODEL_NAME).to(DEVICE)
processor = model.processor

train_dataset = CadenzaDataset(split_dataset["train"], processor)
valid_dataset = CadenzaDataset(split_dataset["validation"], processor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS)

criterion = nn.MSELoss()
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()

# IMPROVEMENT: Add a learning rate scheduler to reduce LR on validation plateau
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1, verbose=True)

print(f"Setup complete. Training with effective batch size of {BATCH_SIZE * ACCUMULATION_STEPS}")
print(f"DataLoader workers set to {NUM_WORKERS} to ensure stability.")

Using device: cuda
Setup complete. Training with effective batch size of 64
DataLoader workers set to 0 to ensure stability.


/tmp/ipykernel_48/3284206543.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [18]:
# Cell 7: Run the Training & Evaluation Loop (Improved with Early Stopping)
import numpy as np
from tqdm.auto import tqdm
import math

# --- IMPROVEMENT: Early Stopping Parameters ---
best_val_rmse = float('inf')
patience_counter = 0
MAX_PATIENCE = 3 # Stop if validation RMSE doesn't improve for 3 consecutive epochs

for epoch in range(EPOCHS):
    model.train()
    model.regression_head.train()
    model.hearing_loss_embedding.train()
    
    running_loss = 0.0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Training]")
    optimizer.zero_grad()
    
    for i, batch in enumerate(train_pbar):
        input_features = batch['input_features'].to(DEVICE)
        hearing_loss_label = batch['hearing_loss_label'].to(DEVICE)
        correctness_score = batch['correctness_score'].to(DEVICE)

        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(input_features, hearing_loss_label)
            loss = criterion(outputs, correctness_score)
            loss = loss / ACCUMULATION_STEPS
        
        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUMULATION_STEPS
        
        if (i + 1) % ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        train_pbar.set_postfix({'loss': f'{loss.item() * ACCUMULATION_STEPS:.4f}'})

    avg_train_loss = running_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []

    valid_pbar = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]")
    with torch.no_grad():
        for batch in valid_pbar:
            input_features = batch['input_features'].to(DEVICE)
            hearing_loss_label = batch['hearing_loss_label'].to(DEVICE)
            correctness_score = batch['correctness_score'].to(DEVICE)
            
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(input_features, hearing_loss_label)
                loss = criterion(outputs, correctness_score)
                
            val_loss += loss.item()
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(correctness_score.cpu().numpy())

    avg_val_loss = val_loss / len(valid_loader)
    all_preds, all_labels = np.concatenate(all_preds), np.concatenate(all_labels)
    val_rmse = math.sqrt(np.mean((all_preds - all_labels)**2))

    print(f"\nEpoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val MSE: {avg_val_loss:.4f} | Val RMSE: {val_rmse:.4f}")

    # IMPROVEMENT: Early Stopping and Model Checkpointing
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        torch.save(model.state_dict(), "best_whisper_regression_model.pth")
        patience_counter = 0
        print(f"  ✓ New best model saved with validation RMSE: {best_val_rmse:.4f}")
    else:
        patience_counter += 1
        print(f"  ✗ No improvement (Patience: {patience_counter}/{MAX_PATIENCE})")
        if patience_counter >= MAX_PATIENCE:
            print(f"\nEarly stopping triggered after {epoch+1} epochs.")
            break
            
    # IMPROVEMENT: Step the learning rate scheduler
    scheduler.step(val_rmse)

print("\nTraining complete!")
print(f"Best validation RMSE achieved: {best_val_rmse:.4f}")

Epoch 1/10 [Training]:   0%|          | 0/991 [00:00<?, ?it/s]

Epoch 1/10 [Validation]:   0%|          | 0/56 [00:00<?, ?it/s]


Epoch 1/10 | Train Loss: 0.1131 | Val MSE: 0.0745 | Val RMSE: 0.2733
  ✓ New best model saved with validation RMSE: 0.2733


Epoch 2/10 [Training]:   0%|          | 0/991 [00:00<?, ?it/s]

Epoch 2/10 [Validation]:   0%|          | 0/56 [00:00<?, ?it/s]

...

Epoch 5/10 [Training]:   0%|          | 0/991 [00:00<?, ?it/s]

Epoch 5/10 [Validation]:   0%|          | 0/56 [00:00<?, ?it/s]


Epoch 5/10 | Train Loss: 0.0839 | Val MSE: 0.0702 | Val RMSE: 0.2656
  ✗ No improvement (Patience: 1/3)


Epoch 6/10 [Training]:   0%|          | 0/991 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [19]:
# Cell 8: Generate Submission File
import pandas as pd
from tqdm.auto import tqdm
import numpy as np

print("Loading the best model checkpoint for inference...")
# Re-initialize model architecture and load the saved weights
inference_model = WhisperRegressionModel(model_name=MODEL_NAME).to(DEVICE)
inference_model.load_state_dict(torch.load("best_whisper_regression_model.pth"))
inference_model.eval()

# We need a new dataloader for the *actual* validation set provided by the competition
# For this example, we'll generate predictions on our own validation split
print("Generating predictions on the validation set...")
all_signal_ids = []
all_predictions = []

with torch.no_grad():
    for batch in tqdm(valid_loader, desc="Generating Predictions"):
        input_features = batch['input_features'].to(DEVICE)
        hearing_loss_label = batch['hearing_loss_label'].to(DEVICE)
        
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            outputs = inference_model(input_features, hearing_loss_label)
        
        # Predictions are scaled from [0, 1] to [0, 100] for submission
        all_predictions.extend(outputs.cpu().numpy().flatten() * 100)
        all_signal_ids.extend(batch['signal'])

# Create and save the submission DataFrame
submission_df = pd.DataFrame({
    'signal_id': all_signal_ids,
    'intelligibility_score': all_predictions
})
submission_df.to_csv('submission.csv', index=False)

print("\nSubmission file 'submission.csv' created successfully!")
print(submission_df.head())

Loading the best model checkpoint for inference...
Generating predictions on the validation set...


Generating Predictions:   0%|          | 0/56 [00:00<?, ?it/s]


Submission file 'submission.csv' created successfully!
                  signal_id  intelligibility_score
0  0a15c1bb98ae10f286529cc1               38.53125
1  f4dc5cea9bf56925f0125d05               54.18750
2  6be3f96b21428fe97e1a5fc5               22.46875
3  52a3d1987e985d2a1e8dbb3c               17.40625
4  b2265a2cc6e44f663442254d               61.90625
